# 21 仮説検証

| 項目 | 内容 |
|------|------|
| プロジェクトID | BAXX-XXX-XX |
| 入力データ | `processed_data/` |
| 作成者 | |
| 更新日 | YYYY-MM-DD |

## 検証する仮説
<!-- 仮説を具体的に記述する。「〜のほうが〜より大きい」という形で明示する -->

**帰無仮説（H₀）**:  
**対立仮説（H₁）**:  
**有意水準（α）**: 0.05

## 検定手法の選択
```
データの種類
├── 数値データ
│   ├── 正規分布に従う
│   │   ├── 2群の平均比較 → t検定（対応なし/あり）
│   │   └── 3群以上 → 一元配置分散分析（ANOVA）
│   └── 正規分布に従わない / サンプル数少ない
│       ├── 2群 → Mann-Whitney U検定 / Wilcoxon符号順位検定
│       └── 3群以上 → Kruskal-Wallis検定
└── カテゴリデータ
    └── 独立性の検定 → カイ二乗検定 / Fisherの正確検定
```

## 結論メモ
<!-- 検証後に記入: p値・効果量・解釈・実務的含意 -->

---
## 0. 環境セットアップ

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib
from scipy import stats

sys.path.append(str(Path('../../').resolve()))
from funcs import data_loader, visualization

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
%matplotlib inline

ALPHA = 0.05  # 有意水準

In [ ]:
cfg      = data_loader.load_yaml('../../configs/configs.yaml')
viz_cfg  = visualization.setup_matplotlib('../../configs/visualize.yaml', theme='notebook')

PROCESSED_DIR = Path(cfg['paths']['processed_data'])
FIGURES_DIR   = Path(cfg['paths']['figures'])

---
## 1. データ読み込み・グループ定義

In [ ]:
# df = data_loader.load_parquet(PROCESSED_DIR / 'processed.parquet')

# 比較するグループを定義
# GROUP_COL = 'group'   # グループを示す列（例: 施策実施有無、地域区分）
# VALUE_COL = 'value'   # 検証したい数値

# group_a = df[df[GROUP_COL] == 'A'][VALUE_COL].dropna()
# group_b = df[df[GROUP_COL] == 'B'][VALUE_COL].dropna()

# print(f'グループA: n={len(group_a)}, mean={group_a.mean():.2f}, std={group_a.std():.2f}')
# print(f'グループB: n={len(group_b)}, mean={group_b.mean():.2f}, std={group_b.std():.2f}')

---
## 2. 分布の確認

In [ ]:
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# axes[0].hist(group_a, bins=30, alpha=0.7, label='グループA')
# axes[0].hist(group_b, bins=30, alpha=0.7, label='グループB')
# axes[0].legend()
# axes[0].set_title('分布比較')

# axes[1].boxplot([group_a, group_b], labels=['グループA', 'グループB'])
# axes[1].set_title('箱ひげ図')
# plt.tight_layout()

---
## 3. 正規性の確認

In [ ]:
# Shapiro-Wilk検定（サンプル数が少ない場合に有効、n<2000推奨）
# stat_a, p_a = stats.shapiro(group_a)
# stat_b, p_b = stats.shapiro(group_b)
# print(f'グループA: W={stat_a:.4f}, p={p_a:.4f}  → {"正規分布に従う" if p_a > ALPHA else "非正規"}')
# print(f'グループB: W={stat_b:.4f}, p={p_b:.4f}  → {"正規分布に従う" if p_b > ALPHA else "非正規"}')

In [ ]:
# Q-Qプロット
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# stats.probplot(group_a, plot=axes[0])
# axes[0].set_title('Q-Qプロット（グループA）')
# stats.probplot(group_b, plot=axes[1])
# axes[1].set_title('Q-Qプロット（グループB）')
# plt.tight_layout()

---
## 4. 統計的検定

In [ ]:
# --- パターンA: t検定（正規分布に従う場合）---

# 等分散性の確認（Levene検定）
# levene_stat, levene_p = stats.levene(group_a, group_b)
# equal_var = levene_p > ALPHA
# print(f'Levene検定: p={levene_p:.4f}  → {"等分散" if equal_var else "不等分散"}')

# Welchのt検定（等分散でない場合も対応）
# t_stat, p_value = stats.ttest_ind(group_a, group_b, equal_var=equal_var)
# print(f't検定: t={t_stat:.4f}, p={p_value:.4f}')
# print(f'→ {"帰無仮説を棄却（有意差あり）" if p_value < ALPHA else "帰無仮説を棄却できない"}')

In [ ]:
# --- パターンB: Mann-Whitney U検定（非正規・ノンパラメトリック）---
# u_stat, p_value = stats.mannwhitneyu(group_a, group_b, alternative='two-sided')
# print(f'Mann-Whitney U検定: U={u_stat:.4f}, p={p_value:.4f}')
# print(f'→ {"帰無仮説を棄却（有意差あり）" if p_value < ALPHA else "帰無仮説を棄却できない"}')

In [ ]:
# --- パターンC: カイ二乗検定（カテゴリ×カテゴリ）---
# contingency_table = pd.crosstab(df['col_a'], df['col_b'])
# chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)
# print(f'カイ二乗検定: χ²={chi2:.4f}, dof={dof}, p={p_value:.4f}')
# print(f'→ {"有意な関連あり" if p_value < ALPHA else "有意な関連なし"}')

In [ ]:
# --- パターンD: 対応ありt検定（同一対象の事前事後比較）---
# t_stat, p_value = stats.ttest_rel(before, after)
# print(f'対応ありt検定: t={t_stat:.4f}, p={p_value:.4f}')

---
## 5. 効果量の計算
> p値は有意差の有無を示すが、効果の大きさは示さない。効果量を合わせて報告する。

In [ ]:
# Cohen's d（2群の平均差の効果量）
# 目安: small=0.2, medium=0.5, large=0.8
# def cohen_d(a, b):
#     n1, n2 = len(a), len(b)
#     pooled_std = np.sqrt(((n1-1)*a.std()**2 + (n2-1)*b.std()**2) / (n1+n2-2))
#     return (a.mean() - b.mean()) / pooled_std

# d = cohen_d(group_a, group_b)
# size = 'small' if abs(d) < 0.2 else 'medium' if abs(d) < 0.5 else 'large'
# print(f"Cohen's d = {d:.4f}  → 効果量: {size}")

---
## 6. 信頼区間の可視化

In [ ]:
# 平均値と95%信頼区間を図示
# def mean_ci(data, confidence=0.95):
#     n = len(data)
#     se = stats.sem(data)
#     ci = stats.t.ppf((1 + confidence) / 2, n-1) * se
#     return data.mean(), ci

# mean_a, ci_a = mean_ci(group_a)
# mean_b, ci_b = mean_ci(group_b)

# fig, ax = plt.subplots(figsize=(7, 4))
# ax.bar(['グループA', 'グループB'], [mean_a, mean_b],
#        yerr=[ci_a, ci_b], capsize=8, color=['steelblue', 'coral'], alpha=0.8)
# ax.set_title('平均値と95%信頼区間')
# ax.set_ylabel(VALUE_COL)
# plt.tight_layout()

---
## 7. 結論と解釈

| 項目 | 値 |
|------|----|
| 検定手法 | |
| p値 | |
| 効果量 | |
| 結論 | |

**解釈**:  
<!-- p値・効果量・サンプルサイズを踏まえた実務的な解釈を記述 -->
<!-- 「統計的に有意」と「実務的に意味がある」は異なることに注意 -->

**限界**:  
<!-- この検証で判断できないこと（交絡・選択バイアス等）を明示 -->